# Playground — эксперименты с Q-learning

Три режима усложнения:
- **Режим 1** — случайный старт + случайная цель каждый эпизод
- **Режим 2** — фиксированный старт/цель, но цель двигается каждые N шагов
- **Режим 3** — всё вместе

In [ ]:
import numpy as np
import random
import copy
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from collections import defaultdict

from env import TankEnv, DEFAULT_MAP
from playground_env import TankEnvEx

MY_MAP = np.array([
    [1,1,1,1,1,1,1,1,1,1],
    [1,0,0,0,1,0,0,0,0,1],
    [1,0,1,0,1,0,1,1,0,1],
    [1,0,1,0,0,0,0,1,0,1],
    [1,0,1,1,1,1,0,1,0,1],
    [1,0,0,0,0,1,0,0,0,1],
    [1,1,1,0,1,1,1,0,0,1],
    [1,0,0,0,1,0,0,0,0,1],
    [1,0,1,1,1,0,1,0,0,1],
    [1,1,1,1,1,1,1,1,1,1],
], dtype=np.int8)

REWARDS = {'step': -2, 'wall': -1, 'closer': +5, 'farther': -5, 'goal': +100}

In [ ]:
class QLearningAgent:
    def __init__(self, n_actions=4, alpha=0.1, gamma=0.95):
        self.n_actions = n_actions
        self.alpha     = alpha
        self.gamma     = gamma
        self.Q         = defaultdict(lambda: np.zeros(self.n_actions, dtype=np.float32))

    def act(self, state, epsilon, rng):
        if rng.random() < epsilon:
            return rng.randrange(self.n_actions)
        q    = self.Q[state]
        best = np.flatnonzero(q == np.max(q))
        return int(rng.choice(best))

    def update(self, s, a, r, s2, done):
        target      = r if done else r + self.gamma * float(np.max(self.Q[s2]))
        self.Q[s][a] += self.alpha * (target - self.Q[s][a])

In [ ]:
def train(env, episodes=10000, seed=42, eps_start=1.0, eps_end=0.05,
          eps_decay=0.999, log_every=1000, **agent_kwargs):
    rng   = random.Random(seed)
    agent = QLearningAgent(**agent_kwargs)
    eps   = eps_start
    rewards_log, win_count = [], 0

    for ep in range(1, episodes + 1):
        obs, done, total = env.reset(), False, 0
        while not done:
            a             = agent.act(obs, eps, rng)
            obs2, r, done = env.step(a)
            agent.update(obs, a, r, obs2, done)
            obs, total    = obs2, total + r
        rewards_log.append(total)
        win_count += (r >= 100)
        eps = max(eps_end, eps * eps_decay)
        if ep % log_every == 0:
            wr = win_count / log_every
            print(f'ep={ep:6d} | eps={eps:.3f} | avg={np.mean(rewards_log[-log_every:]):8.1f} | win={wr:.2f}')
            win_count = 0

    return agent, rewards_log


def record(env, agent, seed=0):
    rng, obs, done = random.Random(seed), env.reset(), False
    frames = [env.render()]
    while not done:
        obs, _, done = env.step(agent.act(obs, 0.0, rng))
        frames.append(env.render())
    return frames


def animate(frames_dict, interval=150):
    labels  = list(frames_dict.keys())
    max_len = max(len(f) for f in frames_dict.values())
    padded  = {k: v + [v[-1]] * (max_len - len(v)) for k, v in frames_dict.items()}

    fig, axes = plt.subplots(1, len(labels), figsize=(5 * len(labels), 5))
    if len(labels) == 1: axes = [axes]
    ims = [ax.imshow(padded[lb][0]) or ax.axis('off') or ax.set_title(lb)
           for ax, lb in zip(axes, labels)]
    ims = [ax.images[0] for ax in axes]

    def upd(i):
        for im, lb in zip(ims, labels):
            im.set_data(padded[lb][i])
        return ims

    ani = animation.FuncAnimation(fig, upd, frames=max_len, interval=interval, blit=True)
    plt.tight_layout()
    plt.close(fig)
    return HTML(ani.to_jshtml())

---
## Режим 1 — случайный старт и цель

Каждый эпизод агент стартует из случайной клетки и ищет случайную цель.  
Q-таблица должна научиться навигации в общем случае, а не для одного маршрута.

In [ ]:
env1 = TankEnvEx(grid=MY_MAP, random_start=True, random_goal=True,
                 rewards=REWARDS, max_steps=300, seed=0)

agent1, rewards1 = train(env1, episodes=15000, eps_decay=0.9995, log_every=1000)
print(f'Состояний в Q-таблице: {len(agent1.Q)}')

In [ ]:
# График обучения
w = 200
sm = np.convolve(rewards1, np.ones(w)/w, mode='valid')
plt.figure(figsize=(10, 4))
plt.plot(rewards1, alpha=0.2, color='steelblue')
plt.plot(range(w-1, len(rewards1)), sm, color='darkorange', label=f'avg-{w}')
plt.axhline(0, color='gray', lw=0.8, ls='--')
plt.title('Режим 1: случайный старт + цель')
plt.xlabel('Эпизод'); plt.ylabel('Reward')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Анимация — фиксируем старт/цель для наглядности
env1_vis = TankEnv(grid=MY_MAP, tank_start=(1,1), goal_pos=(8,7), rewards=REWARDS)
animate({'Режим 1 (random start+goal)': record(env1_vis, agent1)})

---
## Режим 2 — движущаяся цель

Цель перепрыгивает в случайную клетку каждые `goal_move_every` шагов.  
Агент должен не просто идти по маршруту, а постоянно реагировать на изменения.

In [ ]:
env2 = TankEnvEx(grid=MY_MAP, tank_start=(1,1), goal_pos=(8,7),
                 goal_move_every=10, rewards=REWARDS, max_steps=300, seed=1)

agent2, rewards2 = train(env2, episodes=15000, eps_decay=0.9995, log_every=1000)

In [ ]:
w = 200
sm = np.convolve(rewards2, np.ones(w)/w, mode='valid')
plt.figure(figsize=(10, 4))
plt.plot(rewards2, alpha=0.2, color='steelblue')
plt.plot(range(w-1, len(rewards2)), sm, color='darkorange', label=f'avg-{w}')
plt.axhline(0, color='gray', lw=0.8, ls='--')
plt.title('Режим 2: движущаяся цель (каждые 10 шагов)')
plt.xlabel('Эпизод'); plt.ylabel('Reward')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
env2_vis = TankEnvEx(grid=MY_MAP, tank_start=(1,1), goal_pos=(8,7),
                     goal_move_every=10, rewards=REWARDS, seed=42)
animate({'Режим 2 (moving goal)': record(env2_vis, agent2)})

---
## Режим 3 — всё вместе

Случайный старт + случайная цель + цель двигается. Максимальная сложность.

In [ ]:
env3 = TankEnvEx(grid=MY_MAP, random_start=True, random_goal=True,
                 goal_move_every=15, rewards=REWARDS, max_steps=400, seed=2)

agent3, rewards3 = train(env3, episodes=20000, eps_decay=0.9998,
                          eps_end=0.1, log_every=1000)

In [ ]:
w = 200
sm = np.convolve(rewards3, np.ones(w)/w, mode='valid')
plt.figure(figsize=(10, 4))
plt.plot(rewards3, alpha=0.2, color='steelblue')
plt.plot(range(w-1, len(rewards3)), sm, color='darkorange', label=f'avg-{w}')
plt.axhline(0, color='gray', lw=0.8, ls='--')
plt.title('Режим 3: random start+goal + moving goal')
plt.xlabel('Эпизод'); plt.ylabel('Reward')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
env3_vis = TankEnvEx(grid=MY_MAP, tank_start=(1,1), goal_pos=(8,7),
                     goal_move_every=15, rewards=REWARDS, seed=7)
animate({'Режим 3 (всё вместе)': record(env3_vis, agent3)})

---
## Сравнение всех режимов

---
## Режим 4 — Охотник и жертва

Цель каждый шаг убегает от танка — уходит в соседнюю клетку максимально далёкую от него.  
Агент должен научиться загонять жертву в угол, а не просто идти напрямую.

In [ ]:
env4 = TankEnvEx(grid=MY_MAP, tank_start=(1,1), goal_pos=(8,7),
                 goal_flees=True, rewards=REWARDS, max_steps=400, seed=3)

agent4, rewards4 = train(env4, episodes=20000, eps_decay=0.9997,
                          eps_end=0.05, log_every=1000)
print(f'Состояний в Q-таблице: {len(agent4.Q)}')

In [ ]:
w = 200
sm = np.convolve(rewards4, np.ones(w)/w, mode='valid')
plt.figure(figsize=(10, 4))
plt.plot(rewards4, alpha=0.2, color='steelblue')
plt.plot(range(w-1, len(rewards4)), sm, color='darkorange', label=f'avg-{w}')
plt.axhline(0, color='gray', lw=0.8, ls='--')
plt.title('Режим 4: охотник и жертва')
plt.xlabel('Эпизод'); plt.ylabel('Reward')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
env4_vis = TankEnvEx(grid=MY_MAP, tank_start=(1,1), goal_pos=(8,7),
                     goal_flees=True, rewards=REWARDS, seed=42)
animate({'Режим 4 (охотник-жертва)': record(env4_vis, agent4)})

In [ ]:
env_fixed = TankEnv(grid=MY_MAP, tank_start=(1,1), goal_pos=(8,7), rewards=REWARDS)

frames_cmp = {
    'Режим 1 (rand start+goal)': record(env_fixed, agent1),
    'Режим 2 (moving goal)':     record(TankEnv(grid=MY_MAP, tank_start=(1,1), goal_pos=(8,7), goal_move_every=10, rewards=REWARDS, seed=42), agent2),
    'Режим 3 (всё вместе)':      record(TankEnv(grid=MY_MAP, tank_start=(1,1), goal_pos=(8,7), goal_move_every=15, rewards=REWARDS, seed=7),  agent3),
}

for name, frames in frames_cmp.items():
    print(f'{name}: {len(frames)-1} шагов')

animate(frames_cmp, interval=200)